In [1]:
import pandas as pd

df = pd.read_csv("processed.tsv", sep="\t")

In [2]:
import librosa

audio_features = []
for index, row in df.iterrows():
    file_path = f"{row["path"]}"
    audio, sr = librosa.load(file_path, sr=None)

    features = {}
    # 1. MFCC (Mel-Frequency Cepstral Coefficients)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    features['MFCC'] = mfcc

    # 2. Chroma Features
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    features['Chroma'] = chroma

    # 3. Spectral Centroid
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    features['Spectral Centroid'] = spectral_centroid

    # 4. Spectral Rolloff
    spectral_rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)
    features['Spectral Rolloff'] = spectral_rolloff

    # 5. Zero-Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(audio)
    features['Zero Crossing Rate'] = zcr

    # 6. Spectral Bandwidth
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)
    features['Spectral Bandwidth'] = spectral_bandwidth

    audio_features.append(features)


In [3]:
import torch

tensors = []
for audio_feature in audio_features:
    processed_feature = []
    for values in audio_feature.values():
        for value in values:
            processed_feature.append(value)
    feature_tensor = torch.tensor(processed_feature, dtype=torch.float32)
    tensors.append(feature_tensor)


/tmp/ipykernel_4418/2494449280.py:9: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647378361/work/torch/csrc/utils/tensor_new.cpp:278.)
  feature_tensor = torch.tensor(processed_feature, dtype=torch.float32)


In [4]:
torch.save(tensors, "audio_tensors.pt")